# pygraphistry Version Bisect: Find the Breaking Version

This notebook tests multiple pygraphistry versions to identify exactly which
version introduced the SSO/iframe regression.

**Known from local analysis:**
- 0.45.9 = last working version (per client)
- 0.45.10 = first version with `active_organization` required + `_switch_org()`
- 0.47.0 = API version locked to v3 only

**Key insight:** The `viztoken` in plot URLs is a client-generated UUID, NOT a server
auth token. The Graphistry server always requires authentication. The Databricks iframe
works because the browser forwards session cookies from the SSO login. So the real test
is whether **SSO login completes without exceptions** — if it does, the iframe will work.

**Test methodology (corrected):**
1. Install each version
2. SSO login — record whether `sso_get_token()` succeeds or throws
3. If SSO succeeds, upload a test graph and render the iframe for visual verification
4. Record the specific error if SSO fails

**Instructions:** Run cells sequentially. Each version test requires a Python restart
and an SSO login (click the link).

**Target server:** Configure in Cell 1 below.

In [ ]:
# Cell 1: Shared config — run this first, values persist across restarts via Spark conf
# Change these to match your environment:
spark.conf.set("test.server", "graphistry-dev.grph.xyz")
spark.conf.set("test.protocol", "https")

# Versions to test (key boundary versions)
# 0.45.9  = last known working
# 0.45.10 = first with active_org required + _switch_org
# 0.46.0  = same breaking changes, next minor
# 0.47.0  = API version locked to v3
# 0.50.5  = version client reported broken
spark.conf.set("test.versions", "0.45.9,0.45.10,0.46.0,0.47.0,0.50.5")

print("Config saved to Spark conf (persists across Python restarts).")
print(f"Server:   {spark.conf.get('test.server')}")
print(f"Versions: {spark.conf.get('test.versions')}")

---
## Static Analysis (no server needed)

Cell 2 installs all versions one at a time and checks their code structure
without connecting to any server. This identifies which versions have the
breaking changes.

In [ ]:
# Cell 2: Static analysis — install each version and inspect code
# This does NOT require SSO login or server access.
import subprocess, sys, importlib, json

versions = spark.conf.get("test.versions").split(",")
static_results = []

for version in versions:
    print(f"\n{'='*60}")
    print(f"Installing graphistry=={version}...")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"graphistry=={version}"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f"  INSTALL FAILED: {r.stderr[:200]}")
        static_results.append({"version": version, "install": "FAILED"})
        continue

    # Force reimport
    for mod in list(sys.modules.keys()):
        if mod.startswith("graphistry"):
            del sys.modules[mod]

    result = {"version": version, "install": "OK"}

    try:
        import inspect
        from graphistry.arrow_uploader import ArrowUploader

        # Check 1: active_organization required?
        src = inspect.getsource(ArrowUploader.sso_get_token)
        active_org_required = "raise Exception" in src and "active_organization" in src
        active_org_optional = "active_organization" in src and not active_org_required
        result["active_org"] = "REQUIRED (raises)" if active_org_required else "optional (silent)"

        # Check 2: _switch_org exists?
        result["switch_org"] = hasattr(ArrowUploader, "_switch_org")

        # Check 3: API version
        from graphistry.client_session import ClientSession, ApiVersion
        cs = ClientSession()
        result["api_version_type"] = str(ApiVersion)
        result["api_version_default"] = cs.api_version

        # Check 4: maybe_post_share_link guard
        src2 = inspect.getsource(ArrowUploader.maybe_post_share_link)
        result["share_link_guard"] = "session_privacy is not None or g._privacy is not None" in src2

    except Exception as e:
        result["error"] = str(e)

    static_results.append(result)

    status = "LIKELY BREAKS" if result.get("active_org", "").startswith("REQUIRED") else "LIKELY OK"
    print(f"  Version:          {version}")
    print(f"  active_org:       {result.get('active_org', '?')}")
    print(f"  _switch_org:      {result.get('switch_org', '?')}")
    print(f"  ApiVersion:       {result.get('api_version_type', '?')}")
    print(f"  api default:      {result.get('api_version_default', '?')}")
    print(f"  share_link guard: {result.get('share_link_guard', '?')}")
    print(f"  Prediction:       {status}")

# Store results in spark conf for later cells
spark.conf.set("test.static_results", json.dumps(static_results))
print(f"\n\nStored {len(static_results)} results.")

In [ ]:
# Cell 3: Static analysis summary table
import json

static_results = json.loads(spark.conf.get("test.static_results"))

print(f"{'Version':<10} {'active_org':<22} {'_switch_org':<13} {'ApiVersion':<22} {'Default':<9} {'Prediction'}")
print("-" * 95)

for r in static_results:
    if r.get("install") == "FAILED":
        print(f"{r['version']:<10} INSTALL FAILED")
        continue
    pred = "BREAKS" if r.get("active_org", "").startswith("REQUIRED") else "OK"
    print(f"{r['version']:<10} {r.get('active_org','?'):<22} {str(r.get('switch_org','?')):<13} "
          f"{r.get('api_version_type','?'):<22} {str(r.get('api_version_default','?')):<9} {pred}")

print()
print("Conclusion: The breaking version is the first with active_org=REQUIRED.")

---
## Live Test: 0.45.9 (expected: WORKS)

Install 0.45.9, SSO login, upload graph, check iframe.

In [ ]:
# Cell 4: Install 0.45.9 + restart
%pip install graphistry==0.45.9 requests
dbutils.library.restartPython()

In [ ]:
# Cell 5: SSO login with 0.45.9
import graphistry
SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
print(f"graphistry {graphistry.__version__} -> {PROTOCOL}://{SERVER}")

graphistry.register(
    api=3, protocol=PROTOCOL, server=SERVER,
    is_sso_login=True, sso_opt_into_type="display", sso_timeout=None,
)
print("Click the SSO link above, then run the next cell.")

In [ ]:
# Cell 6: Test with 0.45.9 — SSO + plot
import graphistry, pandas as pd, json, traceback

SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
VERSION = graphistry.__version__

result = {"version": VERSION, "sso_ok": False, "plot_ok": False}

# Step 1: SSO token retrieval
try:
    token = graphistry.sso_get_token()
    assert token and len(token) > 10, f"Token empty or too short: {repr(token[:20] if token else None)}"
    result["sso_ok"] = True
    result["token_prefix"] = token[:20]
    print(f"SSO: OK — token={token[:20]}...")
except Exception as e:
    result["sso_error"] = str(e)
    print(f"SSO: FAILED — {e}")
    traceback.print_exc()

# Step 2: Upload + render (only if SSO succeeded)
if result["sso_ok"]:
    try:
        edges = pd.DataFrame({"s": ["a","b","c","d","e"], "d": ["b","c","d","e","a"]})
        g = graphistry.edges(edges, "s", "d")
        url = g.plot(render=False)
        result["plot_ok"] = True
        result["url"] = url
        print(f"Plot: OK — {url}")

        # Render iframe for visual check
        html = f'''
        <h3 style="color:#2d7d2d">v{VERSION}: SSO OK, plot uploaded</h3>
        <p>If the iframe below shows a visualization (not a login page), this version works end-to-end.</p>
        <iframe src="{url}" width="800" height="400" style="border:1px solid #ccc"></iframe>
        <p style="font-size:10px">{url}</p>
        '''
        try:
            displayHTML(html)
        except NameError:
            from IPython.display import display, HTML
            display(HTML(html))
    except Exception as e:
        result["plot_error"] = str(e)
        print(f"Plot: FAILED — {e}")
        traceback.print_exc()

# Step 3: Inspect internal state
print(f"\n--- Internal State ---")
session = graphistry.PyGraphistry._config
print(f"  org_name: {getattr(session, 'org_name', 'MISSING')}")
print(f"  api_version: {getattr(session, 'api_version', 'MISSING')}")
if hasattr(session, '_last_switched_org_token'):
    print(f"  _last_switched_org_token: {session._last_switched_org_token}")

# Store result
try:
    live = json.loads(spark.conf.get("test.live_results"))
except:
    live = {}
live[VERSION] = result
spark.conf.set("test.live_results", json.dumps(live))

# Summary
status = "PASS" if result["sso_ok"] and result["plot_ok"] else "FAIL"
print(f"\n{'='*60}")
print(f"v{VERSION}: {status} (sso={result['sso_ok']}, plot={result['plot_ok']})")

---
## Live Test: 0.45.10 (expected: BREAKS — first breaking version)

Install 0.45.10, SSO login, upload graph, check iframe.

In [ ]:
# Cell 7: Install 0.45.10 + restart
%pip install graphistry==0.45.10 requests
dbutils.library.restartPython()

In [ ]:
# Cell 8: SSO login with 0.45.10
import graphistry
SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
print(f"graphistry {graphistry.__version__} -> {PROTOCOL}://{SERVER}")

graphistry.register(
    api=3, protocol=PROTOCOL, server=SERVER,
    is_sso_login=True, sso_opt_into_type="display", sso_timeout=None,
)
print("Click the SSO link above, then run the next cell.")

In [ ]:
# Cell 9: Test with 0.45.10 — SSO + plot
import graphistry, pandas as pd, json, traceback

SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
VERSION = graphistry.__version__

result = {"version": VERSION, "sso_ok": False, "plot_ok": False}

# Step 1: SSO token retrieval
try:
    token = graphistry.sso_get_token()
    assert token and len(token) > 10, f"Token empty or too short: {repr(token[:20] if token else None)}"
    result["sso_ok"] = True
    result["token_prefix"] = token[:20]
    print(f"SSO: OK — token={token[:20]}...")
except Exception as e:
    result["sso_error"] = str(e)
    print(f"SSO: FAILED — {e}")
    traceback.print_exc()

# Step 2: Upload + render (only if SSO succeeded)
if result["sso_ok"]:
    try:
        edges = pd.DataFrame({"s": ["a","b","c","d","e"], "d": ["b","c","d","e","a"]})
        g = graphistry.edges(edges, "s", "d")
        url = g.plot(render=False)
        result["plot_ok"] = True
        result["url"] = url
        print(f"Plot: OK — {url}")

        html = f'''
        <h3 style="color:#2d7d2d">v{VERSION}: SSO OK, plot uploaded</h3>
        <p>If the iframe below shows a visualization (not a login page), this version works end-to-end.</p>
        <iframe src="{url}" width="800" height="400" style="border:1px solid #ccc"></iframe>
        <p style="font-size:10px">{url}</p>
        '''
        try:
            displayHTML(html)
        except NameError:
            from IPython.display import display, HTML
            display(HTML(html))
    except Exception as e:
        result["plot_error"] = str(e)
        print(f"Plot: FAILED — {e}")
        traceback.print_exc()

# Step 3: Inspect internal state
print(f"\n--- Internal State ---")
session = graphistry.PyGraphistry._config
print(f"  org_name: {getattr(session, 'org_name', 'MISSING')}")
print(f"  api_version: {getattr(session, 'api_version', 'MISSING')}")
if hasattr(session, '_last_switched_org_token'):
    print(f"  _last_switched_org_token: {session._last_switched_org_token}")

# Store result
try:
    live = json.loads(spark.conf.get("test.live_results"))
except:
    live = {}
live[VERSION] = result
spark.conf.set("test.live_results", json.dumps(live))

status = "PASS" if result["sso_ok"] and result["plot_ok"] else "FAIL"
print(f"\n{'='*60}")
print(f"v{VERSION}: {status} (sso={result['sso_ok']}, plot={result['plot_ok']})")
if not result["sso_ok"]:
    print(f"  SSO error: {result.get('sso_error', '?')}")
    print(f"  This is likely the 'active_organization required' regression.")

---
## Live Test: 0.50.5 (expected: BREAKS — version client reported)

Install 0.50.5, SSO login, upload graph, check iframe.

In [ ]:
# Cell 10: Install 0.50.5 + restart
%pip install graphistry==0.50.5 requests
dbutils.library.restartPython()

In [ ]:
# Cell 11: SSO login with 0.50.5
import graphistry
SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
print(f"graphistry {graphistry.__version__} -> {PROTOCOL}://{SERVER}")

graphistry.register(
    api=3, protocol=PROTOCOL, server=SERVER,
    is_sso_login=True, sso_opt_into_type="display", sso_timeout=None,
)
print("Click the SSO link above, then run the next cell.")

In [ ]:
# Cell 12: Test with 0.50.5 — SSO + plot
import graphistry, pandas as pd, json, traceback

SERVER = spark.conf.get("test.server")
PROTOCOL = spark.conf.get("test.protocol")
VERSION = graphistry.__version__

result = {"version": VERSION, "sso_ok": False, "plot_ok": False}

# Step 1: SSO token retrieval
try:
    token = graphistry.sso_get_token()
    assert token and len(token) > 10, f"Token empty or too short: {repr(token[:20] if token else None)}"
    result["sso_ok"] = True
    result["token_prefix"] = token[:20]
    print(f"SSO: OK — token={token[:20]}...")
except Exception as e:
    result["sso_error"] = str(e)
    print(f"SSO: FAILED — {e}")
    traceback.print_exc()

# Step 2: Upload + render (only if SSO succeeded)
if result["sso_ok"]:
    try:
        edges = pd.DataFrame({"s": ["a","b","c","d","e"], "d": ["b","c","d","e","a"]})
        g = graphistry.edges(edges, "s", "d")
        url = g.plot(render=False)
        result["plot_ok"] = True
        result["url"] = url
        print(f"Plot: OK — {url}")

        html = f'''
        <h3 style="color:#2d7d2d">v{VERSION}: SSO OK, plot uploaded</h3>
        <p>If the iframe below shows a visualization (not a login page), this version works end-to-end.</p>
        <iframe src="{url}" width="800" height="400" style="border:1px solid #ccc"></iframe>
        <p style="font-size:10px">{url}</p>
        '''
        try:
            displayHTML(html)
        except NameError:
            from IPython.display import display, HTML
            display(HTML(html))
    except Exception as e:
        result["plot_error"] = str(e)
        print(f"Plot: FAILED — {e}")
        traceback.print_exc()

# Step 3: Inspect internal state
print(f"\n--- Internal State ---")
session = graphistry.PyGraphistry._config
print(f"  org_name: {getattr(session, 'org_name', 'MISSING')}")
print(f"  api_version: {getattr(session, 'api_version', 'MISSING')}")
if hasattr(session, '_last_switched_org_token'):
    print(f"  _last_switched_org_token: {session._last_switched_org_token}")

# Store result
try:
    live = json.loads(spark.conf.get("test.live_results"))
except:
    live = {}
live[VERSION] = result
spark.conf.set("test.live_results", json.dumps(live))

status = "PASS" if result["sso_ok"] and result["plot_ok"] else "FAIL"
print(f"\n{'='*60}")
print(f"v{VERSION}: {status} (sso={result['sso_ok']}, plot={result['plot_ok']})")
if not result["sso_ok"]:
    print(f"  SSO error: {result.get('sso_error', '?')}")
    print(f"  Expected: 'active_organization required' or '_switch_org' failure.")

---
## Final Summary

In [ ]:
# Cell 13: Combined results
import json

print("=" * 80)
print("LIVE TEST RESULTS")
print("=" * 80)

try:
    live = json.loads(spark.conf.get("test.live_results"))
except:
    live = {}

print(f"\n{'Version':<12} {'SSO':<8} {'Plot':<8} {'Error'}")
print("-" * 80)

for ver in sorted(live.keys()):
    r = live[ver]
    sso = "OK" if r.get("sso_ok") else "FAIL"
    plot = "OK" if r.get("plot_ok") else "FAIL"
    err = r.get("sso_error", r.get("plot_error", ""))
    print(f"{ver:<12} {sso:<8} {plot:<8} {err[:50]}")

print()
print("=" * 80)
print("STATIC ANALYSIS RESULTS (from Cell 2)")
print("=" * 80)

try:
    static = json.loads(spark.conf.get("test.static_results"))
    print(f"\n{'Version':<10} {'active_org':<22} {'_switch_org':<13} {'ApiVersion':<22} {'Default'}")
    print("-" * 80)
    for r in static:
        if r.get("install") == "FAILED":
            print(f"{r['version']:<10} INSTALL FAILED")
        else:
            print(f"{r['version']:<10} {r.get('active_org','?'):<22} {str(r.get('switch_org','?')):<13} "
                  f"{r.get('api_version_type','?'):<22} {r.get('api_version_default','?')}")
except:
    print("(Static analysis not run yet — run Cell 2)")

print()
print("=" * 80)
print("CONCLUSION")
print("=" * 80)
print()
print("The viztoken in plot URLs is a client-generated UUID, NOT a server auth token.")
print("The iframe works because the browser forwards session cookies from SSO login.")
print("If SSO login fails, there is no session cookie, and the iframe shows a login page.")
print()
print("Breaking changes introduced in pygraphistry:")
print("  0.45.10: active_organization REQUIRED in SSO response (was optional)")
print("           _switch_org() added — POSTs to /api/v2/o/{org}/switch/")
print("  0.47.0:  API version locked to v3 only (was v1 default)")
print()
print("Recommended: pin to graphistry==0.45.9 until server returns")
print("active_organization in SSO JWT response and supports /api/v2/o/{org}/switch/")